In [1]:
%pip install tiktoken
import pandas as pd
import math
import torch
import random
from random import randrange

from torch import nn
from torch.nn import functional as F
import torch.nn.utils as nn_utils
import tiktoken
import dask.dataframe as dd
from transformers import AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'

Note: you may need to restart the kernel to use updated packages.


In [2]:
reasoning_start = "<start_working_out>" # Acts as <think>
reasoning_end   = "<end_working_out>"   # Acts as </think>
solution_start  = "<SOLUTION>"
solution_end    = "</SOLUTION>"

system_prompt = \
f"""You are given a problem.
Think about the problem and provide your working out.
Place it between {reasoning_start} and {reasoning_end}.
Then, provide your solution between {solution_start}{solution_end}"""
system_prompt

chat_template = \
    "{% if messages[0]['role'] == 'system' %}"\
        "{{ messages[0]['content'] + eos_token }}"\
        "{% set loop_messages = messages[1:] %}"\
    "{% else %}"\
        "{{ '{system_prompt}' + eos_token }}"\
        "{% set loop_messages = messages %}"\
    "{% endif %}"\
    "{% for message in loop_messages %}"\
        "{% if message['role'] == 'user' %}"\
            "{{ message['content'] }}"\
        "{% elif message['role'] == 'assistant' %}"\
            "{{ message['content'] + eos_token }}"\
        "{% endif %}"\
    "{% endfor %}"\
    "{% if add_generation_prompt %}{{ '{reasoning_start}' }}"\
    "{% endif %}"

# Replace with out specific template:
chat_template = chat_template\
    .replace("'{system_prompt}'",   f"'{system_prompt}'")\
    .replace("'{reasoning_start}'", f"'{reasoning_start}'")

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Base")
tokenizer.chat_template = chat_template

In [3]:
message = tokenizer.apply_chat_template([
    {"role" : "user", "content" : "What is 1+1?"},
    {"role" : "assistant", "content" : f"{reasoning_start}I think it's 2.{reasoning_end}{solution_start}2{solution_end}"},
    {"role" : "user", "content" : "What is 2+2?"},
], tokenize = False, add_generation_prompt = True)

token_encoder = tiktoken.get_encoding("cl100k_base")

token_encoder.encode(message, allowed_special = {'<|endoftext|>'})

dataset = pd.read_parquet("../data/open-r1_OpenR1-Math-220k/train-00000-of-00010.parquet")
           
def format_dataset(x):
    expected_answer = x["answer"]
    problem = x["problem"]

    # Remove generated <think> and </think>
    thoughts = x["generations"][0]
    thoughts = thoughts.replace("<think>", "").replace("</think>", "")

    # Strip newlines on left and right
    thoughts = thoughts.strip()
    # Add our custom formatting
    final_prompt = \
        reasoning_start + thoughts + reasoning_end + \
        solution_start + expected_answer + solution_end
    return [
        {"role" : "system",    "content" : system_prompt},
        {"role" : "user",      "content" : problem},
        {"role" : "assistant", "content" : final_prompt},
    ]



# Dot Product Attention

In [4]:
class DotProductAttention(nn.Module):
    def __init__(self, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def masked_softmax(self, X: torch.Tensor, valid_lens: torch.Tensor):
        def _sequence_mask(X: torch.Tensor, valid_lens: torch.Tensor, value=0):
            maxlen = X.size(1)
            mask = torch.arange((maxlen), dtype=torch.float32, device=device)[None,:] < valid_lens[:, None]
            X[~mask] = value
            return X

        if valid_lens is None:
            return nn.functional.softmax(X, dim=-1)
        else:
            # print(f'masking : {X.shape} : valid_lens : {valid_lens.shape}')
            shape = X.shape
            if valid_lens.dim() == 1:
                valid_lens = torch.repeat_interleave(valid_lens, shape[1])
            else:
                valid_lens = valid_lens.reshape(-1)
            X = _sequence_mask(X.reshape(-1, shape[-1]), valid_lens, value=-1e6)
            return nn.functional.softmax(X.reshape(shape), dim=-1)

    @torch.compile
    def forward(self, queries: torch.Tensor, keys: torch.Tensor, values: torch.Tensor, valid_lens: torch.Tensor):
        d = queries.shape[-1]
        scores = torch.bmm(queries, keys.transpose(1,2))/math.sqrt(d)
        # print(f'Attention score : {scores.shape}')

        self.attention_weights = self.masked_softmax(scores, valid_lens)
        return torch.bmm(self.dropout(self.attention_weights), values)

# Multi-Head Attention

In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_hiddens, num_heads, dropout, bias=False):
        super().__init__()
        self.num_heads = num_heads
        self.attention = DotProductAttention(dropout)
        self.W_o = nn.LazyLinear(num_hiddens, bias=bias)

    def transpose_qkv(self, X: torch.Tensor):
        X = X.reshape(X.shape[0], X.shape[1], self.num_heads, -1)
        X = X.permute(0,2,1,3)
        X = X.reshape(-1, X.shape[2], X.shape[3])
        return X

    def transpose_output(self, X: torch.Tensor):
        X = X.reshape(-1, self.num_heads, X.shape[1], X.shape[2])
        X = X.permute(0,2,1,3)
        X = X.reshape(X.shape[0], X.shape[1], -1)
        return X

    @torch.compile(mode="max-autotune")
    def forward(self, queries: torch.Tensor, keys: torch.Tensor, values: torch.Tensor, valid_lens: torch.Tensor):
        # print(f'MultiHeadAttention Before : Queries : {queries.shape} : Keys : {keys.shape} : Values : {values.shape} : Valid lens : {valid_lens.shape}')

        queries = self.transpose_qkv(queries)
        keys = self.transpose_qkv(keys)
        values = self.transpose_qkv(values)
        
        # print(f'MultiHeadAttention After : Queries : {queries.shape} : Keys : {keys.shape} : Values : {values.shape} : Valid lens : {valid_lens.shape}')
        if valid_lens is not None:
            valid_lens = torch.repeat_interleave(valid_lens, repeats=self.num_heads, dim=0)

        output = self.attention(queries, keys, values, valid_lens)
        output_concat = self.transpose_output(output)
        return self.W_o(output_concat)

# RMSNorm, MLP, Rope Embedding

In [6]:
class RMSNorm(nn.Module):
    def __init__(self, num_hiddens, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(num_hiddens,device=device))
        self.variance_epsilon = eps
    
    @torch.compile(mode="max-autotune")
    def forward(self, hidden_state: torch.Tensor):
        input_dtype = hidden_state.dtype
        hidden_state = hidden_state.to(torch.float32)
        variance = hidden_state.pow(2).mean(-1, keepdim=True)
        hidden_state = hidden_state*torch.rsqrt(variance+self.variance_epsilon)
        return self.weight*hidden_state.to(input_dtype)


class MLP(nn.Module):
    """The positionwise feed-forward network."""
    def __init__(self, num_hiddens, num_intermediate):
        super().__init__()
        self.num_hiddens = num_hiddens
        self.num_intermediate = num_intermediate
        self.gate_proj = nn.Linear(self.num_hiddens, self.num_intermediate, bias=False)
        self.up_proj = nn.Linear(self.num_hiddens, self.num_intermediate, bias=False)
        self.down_proj = nn.Linear(self.num_intermediate, self.num_hiddens, bias=False)
        self.act_fn = nn.SiLU()

    @torch.compile(mode="max-autotune")
    def forward(self, X):
        down_proj = self.down_proj(self.act_fn(self.gate_proj(X)) * self.up_proj(X))
        return down_proj

class RopeEmbedding(nn.Module):
    def __init__(self, num_hiddens, dropout, m):
        super().__init__()
        # n = 10000 as per paper
        self.n = 10000
        self.dropout = nn.Dropout(dropout)
        # k/n^(2i/d) with n = 10000
        theta = torch.pow(self.n, -2*torch.arange(1,num_hiddens//2+1,dtype=torch.float64,device=device)/num_hiddens)
        expression = torch.arange(1, m+1, dtype=torch.float64,device=device).reshape(-1,1)*theta
        # print(f'Theta : {theta}')
        sin_val = torch.sin(expression)
        cos_val = torch.cos(expression)
        # print(f'sin_val : {sin_val}')
        # print(f'cos_val : {cos_val}')

        self.sin = sin_val.repeat_interleave(2, dim=-1)
        self.cos = cos_val.repeat_interleave(2, dim=-1)
        self.sin = self.sin.unsqueeze(0)
        self.cos = self.cos.unsqueeze(0)

    @torch.compile(mode="max-autotune")
    def forward(self,query,key):
        def rotate_half(x):
            x1 = x[...,:x.shape[-1]:2].unsqueeze(1)
            x2 = x[...,1:x.shape[-1]:2].unsqueeze(1)
            result = torch.cat((-x2,x1), dim=-1).squeeze()
            # print(f'x1: {x1.shape}, x2: {x2.shape}, result: {result.shape}')
            return result
        # print(f'Query : {query.shape} , Cos : {self.cos.shape}, Rotate half : {rotate_half(query).shape}, Sin : {self.sin.shape}')
        q_type = query.dtype
        k_type = key.dtype
        q_embed = (query.float()*self.cos.float()) + (rotate_half(query).float()*self.sin.float())
        k_embed = (key.float()*self.cos.float()) + (rotate_half(key).float()*self.sin.float())
        return q_embed.to(q_type), k_embed.to(k_type)


# TransformerDecoder

In [7]:
# import ast
class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, num_hiddens, mlp_intermediate_hidden, num_heads, dropout, batch_size, block_size, L):
        super().__init__()
        self.L = L
        self.vocab_size = vocab_size
        self.num_hiddens = num_hiddens
        self.batch_size = batch_size
        self.block_size = block_size
        self.embedding = nn.Embedding(vocab_size, num_hiddens)
        self.rope_embedding = RopeEmbedding(num_hiddens, dropout, block_size)
        self.W_q = nn.LazyLinear(num_hiddens, bias=False)
        self.W_k = nn.LazyLinear(num_hiddens, bias=False)
        self.W_v = nn.LazyLinear(num_hiddens, bias=False)
        self.attention = MultiHeadAttention(num_hiddens, num_heads,
                                                 dropout)
        self.RMS1 = RMSNorm(num_hiddens)
        self.RMS2 = RMSNorm(num_hiddens)
        self.MLP = MLP(num_hiddens, mlp_intermediate_hidden)
        self.W_down = nn.LazyLinear(num_hiddens)
        self.RMS3 = RMSNorm(num_hiddens)
        self.dense = nn.LazyLinear(vocab_size)

    @torch.compile(mode="max-autotune")
    def forward(self, X, targets=None, generate=False):
        X = self.embedding(X)
        # print(f'Batch Size : {self.batch_size}')
        valid_lens = torch.repeat_interleave(torch.arange(1,self.block_size+1,device=device).unsqueeze(0), self.batch_size, dim=0)
        if generate:
            valid_lens = torch.repeat_interleave(torch.arange(1,self.block_size+1,device=device).unsqueeze(0), 1, dim=0)

        for iter in range(self.L):
            #RMS1
            Y = self.RMS1(X)
            Q, K, V = self.W_q(Y), self.W_k(Y), self.W_v(Y)
            #rope embedding
            Q,K = self.rope_embedding(Q,K)
            #masked attention
            # if generate:
            ZW_o = self.attention(Q,K,V,valid_lens)
            # else:
            #     ZW_o = self.attention(Q,K,V)
            #residual
            X1 = V + ZW_o
            #RMS2
            Y1 = self.RMS2(X1)
            #MLP/SwiGLU
            Z = self.MLP(Y1)
            #Residual
            X = X1 + self.W_down(Z)
        X = self.RMS3(X)
        logits = self.dense(X)
        logits = logits.float()

        if targets == None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            shift_labels = targets[..., 1:].contiguous()
            shift_logits = logits[...,:-1,:].contiguous()
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(shift_logits, shift_labels)
        return logits, loss

    def probs(self, X):
        logits, _ = self(X, generate=True)
        probs = F.softmax(logits, dim=-1)
        return probs
            

    def generate(self, X, max_new_tokens):
        # valid_lens = torch.repeat_interleave(torch.arange(1,self.block_size+1,device=device).unsqueeze(0), self.batch_size, dim=0)
        for i in range(max_new_tokens):
            logits, loss = self(X, generate=True)
            # logits = logits[:,-1,:]
            probs = F.softmax(logits, dim=-1)
            print(probs.shape)
            X_next = torch.multinomial(probs, num_samples=1) #(B,1)
            print(f'Generate X_next : {X_next.shape}')
            X = torch.cat((X[0][1:].unsqueeze(0), X_next), dim=1) #(B,T+1)
            print(f'Generate X : {X.shape}')
            # if i%100 == 0:
            #     print(f'tokens generated : {i}')
        return X

    @property
    def attention_weights(self):
        return self._attention_weights

# Reference Model

In [8]:
class ReferenceModel:
    
    def __init__(self,model_path):
        self.max_iters = 15000
        self.eval_interval = 100
        self.eval_iter = 5
        self.warmup_steps = 4000
        self.batch_size = 8
        self.target_batch_size = 64 
        self.accumulation_steps = self.target_batch_size // self.batch_size  

        self.block_size = 512
        self.L = 8
        self.num_hiddens, self.dropout = 1024, 0.1
        self.mlp_intermediate_hidden, self.num_heads = 1024, 8
        self.token_encoder = tiktoken.get_encoding("cl100k_base")

        self.model_path = model_path
        model = TransformerDecoder(self.token_encoder.n_vocab, self.num_hiddens,
                                        self.mlp_intermediate_hidden, self.num_heads,
                                        self.dropout, self.batch_size,
                                        self.block_size, self.L)
        self.m = model.to(device)
        self.model_initalized = False
        self.dense = nn.LazyLinear(self.token_encoder.n_vocab)

    
    def intialize(self):
        checkpoint = torch.load(self.model_path, weights_only=True, map_location=torch.device('cpu'))
        self.m.load_state_dict(checkpoint['model_state_dict'])
        self.model_initalized = True
    
    def logits(self, X):
        if not self.model_initalized:
            self.intialize()
        logits, _ = self.m(X)
        return logits
    

# Base runner

Here we should implment the base runner with few things

1. Bookcorpus : To be able to write basic english sentences

2. Template Data : So that LLM can understand instructions

In [9]:
import gc

class Runner:
    # init configurations
    def __init__(self):
        self.max_iters = 10000
        self.eval_interval = 100
        self.eval_iter = 200
        self.warmup_steps = 500
        self.batch_size = 16
        self.target_batch_size = 64 
        self.accumulation_steps = self.target_batch_size // self.batch_size  

        self.block_size = 512
        self.L = 32
        self.num_hiddens, self.dropout = 512, 0.1
        self.mlp_intermediate_hidden, self.num_heads = 1024, 32

        model = TransformerDecoder(token_encoder.n_vocab, self.num_hiddens,
                                        self.mlp_intermediate_hidden, self.num_heads,
                                        self.dropout, self.batch_size,
                                        self.block_size, self.L)
        self.m = model.to(device)
        self.parquet_file_index = 0
        self.optimizer = torch.optim.AdamW(self.m.parameters(), lr=1e-2, weight_decay=0.01)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(self.optimizer, T_0=self.warmup_steps, eta_min=8e-5)
        self.model_save_name = f"book-corpus-model"
        self.book_corpus_index = 0
        self.r1_index = 0
        self.book_corpus_paths = [
                    "../data/book-corpus/0005.parquet",
                    "../data/book-corpus/0008.parquet",
                    # "../data/book-corpus/0003.parquet",
                    # "../data/book-corpus/0004.parquet"
                ]
        self.r1_paths = [
                    "../data/open-r1_OpenR1-Math-220k/train-00000-of-00010.parquet",
                    "../data/open-r1_OpenR1-Math-220k/train-00001-of-00010.parquet",
                    "../data/open-r1_OpenR1-Math-220k/train-00008-of-00010.parquet",
                    "../data/open-r1_OpenR1-Math-220k/train-00009-of-00010.parquet"
                ]
        
    def get_model(self,model_path=""):
        if len(model_path) != 0:
            checkpoint = torch.load(model_path, weights_only=True)
            self.m.load_state_dict(checkpoint['model_state_dict'])
            self.optimizer.load_state_dict(checkpoint['optimizer_state_dic'])
        return self.m

    def get_lr(self):
        for param_group in self.optimizer.param_groups:
            return param_group['lr']
        
    def mix_frames(self, iter, start=False):
        if iter == 0 and start:
            current_path = self.book_corpus_paths[random.randint(0,len(self.book_corpus_paths)-1)]
            # print(f"path change : {current_path}")
            self.book_corpus_df = pd.read_parquet(current_path)
        elif iter == 5000 and start:
            # print(f"path change : synthetic dataset")
            del self.book_corpus_df
            torch.cuda.empty_cache()
            gc.collect()
        elif iter == 7500 and start:
            torch.cuda.empty_cache()
            gc.collect()
            current_path = self.r1_paths[random.randint(0,len(self.r1_paths)-1)]
            # print(f"path change : {current_path}")
            self.open_r1_df = pd.read_parquet(current_path)
            self.open_r1_df["Messages"] = self.open_r1_df.apply(format_dataset, axis = 1)

        def book_corpus_token():
            text = self.book_corpus_df[self.book_corpus_index:self.book_corpus_index+1]["text"]
            tokens = torch.tensor(token_encoder.encode(text[self.book_corpus_index]), dtype=torch.long, device=device)
            self.book_corpus_index += 1
            return tokens
        
        if iter < 5000:
            if self.book_corpus_index < len(self.book_corpus_df):
                return book_corpus_token()
            else:
                current_path = self.book_corpus_paths[random.randint(0,len(self.book_corpus_paths)-1)]
                # print(f"path change : {current_path}")
                self.book_corpus_df = pd.read_parquet(current_path)
                self.book_corpus_index = 0
                return book_corpus_token()
        elif iter >= 5000 and iter < 7500:
            tokens = self.synthetic_math()
            return torch.tensor(tokens, dtype=torch.long, device=device)
        elif iter >= 7500 and iter < 12500:
            if self.r1_index < len(self.open_r1_df):
                text_list = self.open_r1_df["Messages"][self.r1_index]
                text = tokenizer.apply_chat_template(text_list, tokenize = False, add_generation_prompt = True)
                tokens = torch.tensor(token_encoder.encode(text, allowed_special = {'<|endoftext|>'}), dtype=torch.long, device=device)
                self.r1_index += 1
                # print(f'Tokens type : {tokens.shape}')
                return tokens
            else:
                current_path = self.r1_paths[random.randint(0,len(self.r1_paths)-1)]
                # print(f"path change : {current_path}")
                self.open_r1_df = pd.read_parquet(current_path)
                self.open_r1_df["Messages"] = self.open_r1_df.apply(format_dataset, axis = 1)
                self.r1_index = 0
                text_list = self.open_r1_df["Messages"][self.r1_index]
                text = tokenizer.apply_chat_template(text_list, tokenize = False, add_generation_prompt = True)
                tokens = torch.tensor(token_encoder.encode(text, allowed_special = {'<|endoftext|>'}), dtype=torch.long, device=device)
                self.r1_index += 1
                # print(f'Tokens type : {tokens.shape}')
                return tokens
        return torch.tensor([],dtype=torch.long, device=device)

    def synthetic_math(self):
        text_list = []
        for _ in range(100):
            probs = random.randint(1,10)
            if probs < 4:
                a = random.randint(-100,100)
                b = random.randint(-100,100)
            elif probs < 7:
                a = random.randint(-100000,100000)
                b = random.randint(-100000,100000)
            else:
                a = random.random()
                b = random.random()
            operator_list = [['+'],['-'], ['*','x'], ['/']]
            current_index = random.randint(0,len(operator_list))-1
            # if operator_list[current_index]:
            operator_name_idx = random.randint(0,len(operator_list[current_index])-1)
            operator_name = operator_list[current_index][operator_name_idx]

            if '+' in operator_list[current_index]:
                result = a+b
            elif '-' in operator_list[current_index]:
                result = a-b
            elif '*' in operator_list[current_index]:
                result = a*b
            elif '/' in operator_list[current_index]:
                if b == 0:
                    result = "Divisible by zero is undefined"
                else:
                    result = a/b
            else:
                result = "Operation is not supported"
            text_list.append({"role" : "user", "content" : f"What is {a} {operator_name} {b}?"})
            text_list.append({"role" : "assistant", "content" : f"{reasoning_start}I think it's {result}.{reasoning_end}{solution_start}{result}{solution_end}"})
        text = tokenizer.apply_chat_template(text_list, tokenize = False, add_generation_prompt = True)
        return token_encoder.encode(text, allowed_special = {'<|endoftext|>'})
    
    def checkIfNumber(self, number: str):
        try:
            float(number)
            return True
        except ValueError:
            return False

    # batch 
    # @torch.compile
    def get_batch(self, step):
        x_list, y_list = [],[]
        start = True
        for _ in range(self.batch_size):
            while start or data.shape[-1] < self.block_size+1:
                if start:
                    data = self.mix_frames(step, start)
                    start = False
                else:
                    data = torch.cat((data, self.mix_frames(step, start)),0)

            if data is not None and data.shape[0] > self.block_size:
                x_list.append(data[:self.block_size])
                y_list.append(data[1:self.block_size+1])
        x = torch.stack(x_list)
        y = torch.stack(y_list)
        if x is not None and y is not None:
            x, y = x.to(device), y.to(device)
        return x,y

    # @torch.compile
    def execute(self):
        running_loss = torch.zeros([1], dtype=torch.float32, device=device)
        # checkpoint = torch.load(f"./{self.model_save_name}", weights_only=True)
        # self.m.load_state_dict(checkpoint['model_state_dict'])
        for step in range(self.max_iters):
            self.iter = step
            xb, yb = self.get_batch(step)
            # print(f'Shapes : {xb.shape} : {yb.shape}')
            # evaluate the loss
            logits, cross_entropy_loss = self.m(xb, yb)
            cross_entropy_loss = cross_entropy_loss / self.accumulation_steps
            cross_entropy_loss.backward()
            running_loss += cross_entropy_loss.item()*self.accumulation_steps
            print(f"step {step}: train loss={cross_entropy_loss.item()}, lr:{self.get_lr():.5f}")

            if (step + 1) % self.accumulation_steps == 0:
                self.optimizer.step()
                self.scheduler.step()
                self.optimizer.zero_grad(set_to_none=True)
                # print(f"step {step}: train loss={running_loss}, lr:{self.get_lr():.5f}")
                running_loss = torch.zeros([1], dtype=torch.float32, device=device)



            if step % self.eval_interval == 0:
                torch.save({
                    'epoch': step,
                    'model_state_dict': self.m.state_dict(),
                    'optimizer_state_dic': self.optimizer.state_dict(),
                    'loss': cross_entropy_loss
                    }, f"./{self.model_save_name}")
                print(f'save completed at {step}')

In [ ]:
runner = Runner()
runner.execute()

/Users/debashisdas/Projects/DeepLearning/.venv/lib/python3.11/site-packages/torch/nn/modules/lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '
[2025-09-27 09:46:05,262] [0/0] torch._dynamo.output_graph: [WARNING] nn.Module forward/_pre hooks are only partially supported, and were detected in your model. In particular, if you do not change/remove hooks after calling .compile(), you can disregard this warning, and otherwise you may need to set torch._dynamo.config.skip_nnmodule_hook_guards=False to ensure recompiling after changing hooks.See https://pytorch.org/docs/master/compile/nn-module.html for more information and limitations. 
[2025-09-27 09:46:05,263] [0/0] torch._dynamo.output_graph: [WARNING] nn.Module state_dict and backward hooks are not yet supported by torch.compile, but were detected in your model and will